In [1]:
import pandas as pd
import os

python_directory = os.getcwd()

# Manage NetLogo input data

## Manually add stop points for Tillydrone

In [43]:
df = pd.read_csv(rf"{python_directory}/node_list.csv")
## the `stoppoint_id` list should come from an analysis of the closes node in the ABM to each stoppoint
## this was applied manually for Tillydrone, but a gespatial analysis will be added later to allocate the stoppoints systematically before importing to NetLogo
stoppoint_id = [11, 214, 38, 557, 558, 592, 596, 682, 712, 716, 781, 782, 792, 800, 821, 1197]
stoppoint_id.sort()
print(f"List of input stoppoint ids (node ids in the NetLogo input files) = {stoppoint_id}\n")

def allocate_stoppoint(node_id:str, stoppoint_id:list):
    int_node_id = int(node_id.replace("n_id", ""))
    if int_node_id in stoppoint_id: return int_node_id
    else: return -1

df["stoppoint"] = [allocate_stoppoint(id, stoppoint_id) for id in df["id"]]
print("Stoppoints in the dataframe")
display(df[df["stoppoint"] > -1])

List of input stoppoint ids (node ids in the NetLogo input files) = [11, 38, 214, 557, 558, 592, 596, 682, 712, 716, 781, 782, 792, 800, 821, 1197]

Stoppoints in the dataframe


,id,xpos,ypos,stoppoint
11,n_id11,0.638384,0.711153,11
38,n_id38,0.244150,0.666634,38
214,n_id214,0.485509,0.504593,214
557,n_id557,0.902793,0.430520,557
558,n_id558,0.916921,0.393484,558
592,n_id592,0.772165,0.593270,592
596,n_id596,0.835567,0.538426,596
682,n_id682,0.626706,0.619268,682
712,n_id712,0.547933,0.817286,712
716,n_id716,0.292753,0.549466,716


In [44]:
df.to_csv(rf"{python_directory}/node_list.csv", index=False)
print(f"Saved csv file to {python_directory}/node_list.csv")

Saved csv file to c:\Users\yhmg1v\OneDrive - University of Glasgow\Google Drive\10 Coding Models\09 TRIFIC\MATRIFIC\Data/node_list.csv


## Manually create dummy bus input data

This function is designed to work for one bus service to fit the Tillydrone application. The data format generated for the `trips_list.csv` is the target format when analysing the GTFS transport data.

In [2]:
def dummy_bus_data(service:int, frequency:int, sp_id:list, time:list, start:list):
    """
    Generate dummy bus dataframes.

    Parameters
    ----------
    `service`: int
        The service number
    `frequency`: int
        The frequency of the service in minutes
    `sp-id`: list of int
        The ids of the stop points the bus reaches
    `time`: list of int
        The time take to reach each stop point from the previous one in minutes
    `start`: list of int
        Whether the bus starts/terminates at the respective stop point (1 = True, 0 = False)

    Returns
    -------
    Dictionary
        "buses": DataFrame where each entry is a bus
        "trips": DataFrame where each entry is a trip
    """
    ## frequency of the dummy bus (use to control the produced csv)
    f = frequency
    s = service
    ## dummy dataframe where each entry (row) represents a bus
    ## should include the following columns (to be extracted from the GTFS data)
    dict_buses = {
        "service":  [s],    # the number of the service
        "frequency": [f],   # the frequency (in minutes) of the service
        "sp-id": [sp_id],   # the stop points ids (as per the `sp-id` variable in NetLogo, which will be based on the stop points ids in the TransXChange data)
        "time":  [time],    # the time taken to reach the stop point for the respective previous one
        "start": [start]    # whether the bus starts/terminates at the respective stop point or not
    }

    ## dummy dataframe where each entry represents a trip
    ## it is recommended to preprocess a csv file per simulated service to simplify the csv file processing in NetLogo
    dict_trips = {
        "service":   [service for i in range(len(sp_id))],      # the number of the bus service makeing the trip
        "sp-id":     sp_id,     # the stop points ids (as per the `sp-id` variable in NetLogo, which will be based on the stop points ids in the TransXChange data)
        "time":      time,      # the time taken to make the trip to the stop point from the respective previous stop point
        "frequency": [frequency for i in range(len(sp_id))],    # the frequency of this trip being made
        "start":     start
    }

    ## generate dataframes
    df_buses = pd.DataFrame(dict_buses)
    df_trips = pd.DataFrame(dict_trips)

    return_dict = {
        "buses": df_buses,
        "trips": df_trips
    }

    ## save dataframes as csv
    print("Dummy dataframe for all buses")
    display(df_buses)

    print("Dummy dataframe for all trips")
    display(df_trips)

    return return_dict


In [3]:
## generate dummy bus data
df_dict = dummy_bus_data(
    service=19,
    frequency=15,
    sp_id=[1197, 781, 557, 592, 682, 214, 716,  38, 800, 792, 712, 821, 11, 596, 558, 782, 1197],
    time =[   0, 0.5,   1,   1,   1,   1,   1, 0.5, 0.5,   1,   1,   1,  1,   1,   1, 0.5,  0.5],
    start=[   1,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,  0,   0,   0,   0,    0]
)
df_buses = df_dict["buses"]
df_trips = df_dict["trips"]

## save the data to csv
df_buses.to_csv(rf"{python_directory}/buses_list.csv", index=False)
print(f"Saved csv file to {python_directory}/buses_list.csv")
df_trips.to_csv(rf"{python_directory}/trips_list.csv", index=False)
print(f"Saved csv file to {python_directory}/trips_list.csv")

Dummy dataframe for all buses


,service,frequency,sp-id,time,start
0,19,15,"[1197, 781, 557, 592, 682, 214, 716, 38, 800, ...","[0, 0.5, 1, 1, 1, 1, 1, 0.5, 0.5, 1, 1, 1, 1, ...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


Dummy dataframe for all trips


,service,sp-id,time,frequency,start
0,19,1197,0.0,15,1
1,19,781,0.5,15,0
2,19,557,1.0,15,0
3,19,592,1.0,15,0
4,19,682,1.0,15,0
5,19,214,1.0,15,0
6,19,716,1.0,15,0
7,19,38,0.5,15,0
8,19,800,0.5,15,0
9,19,792,1.0,15,0


Saved csv file to c:\Users\yhmg1v\OneDrive - University of Glasgow\Google Drive\10 Coding Models\09 TRIFIC\MATRIFIC\Data/buses_list.csv
Saved csv file to c:\Users\yhmg1v\OneDrive - University of Glasgow\Google Drive\10 Coding Models\09 TRIFIC\MATRIFIC\Data/trips_list.csv


In [ ]:
buses-own [
  service             ; The number of the bus service
  location            ; Current location of the vehicle (origin node on current link)
  destination         ; Destination of the vehicle (destination node on current link)
  location-time       ; Time step at which the vehicle was at the loaction
  destination-time    ; Time step at which the vehicle reached the destination
  top-speed           ; Maximum speed of the vehicle
  speed               ; Current speed of the vehicle
  speed-restriction   ; Local speed restriction
  progress            ; Total distance of the current progress on a road (total distance from origin to reach the destination)
  remaining-progress  ; Remaining distance of the current progress on a road (remaining distance from origin to reach the destination)
  trip                ; the path to next stoppoint destination is a list of roads
  active?             ; if vehicle has not finished all its required trips (reached all its stop points
  at-sp?              ; whether the vehicle is at a stop-point or not
  my-home             ; home of vehicle
  sp-destinations     ; the list of all the node destinations (in order)
  sp-destination      ; the destination building for the vehicle
  i-destination       ; the index of the destination stop point in the sp-destinations
  target-times        ; the target time steps to reach each destination
  times               ; the time steps at which the vehicle reached its destinations
  delays              ; the delay periods (comparing times to target-times)
  expected-periods    ; the expected time steps to be taken to reach the destinations
  periods             ; the list of the periods taken to reach each b-destination
  initial-step?       ; whether the vehicle is at the initial trip from home or not
]

nodes-own[
  n-id ; ID of node
  n-xpos ; x-position of node
  n-ypos ; y-position of node
  sp-id  ; stop point ID of node (-1 if not a stop point)
  buses-id      ; the service number of the buses passing through the node (empty list if not a stop point)
  buses-freq    ; the frequency of the serive (empty list if not a stop point)
  buses-start   ; whether the buses starts from the startpoint or not
  buses-period  ; the period until the next bus arrives (empty list if not a stop point)
]

templates-own [
  service             ; The number of the bus service
  location            ; Current location of the vehicle (origin node on current link)
  active?             ; if vehicle has not finished all its required trips (reached all its stop points
  sp-destinations     ; the list of all the node destinations (in order)
  sp-destination      ; the destination building for the vehicle
  expected-periods    ; the expected time steps to be taken to reach the destinations
]